# CS F425 Deep Learning — Course Project
## Phase 2: ToolAlpaca SFT & The ReAct Execution Loop

### Overview
This notebook implements both parts of Phase 2:
- **Part A**: Supervised Fine-Tuning (SFT) of a 7B language model on the ToolAlpaca dataset using QLoRA, formatted in `Thought → Action → Action Input` style.
- **Part B**: A pure-Python ReAct (Reason + Act) execution loop with custom stopping criteria, robust JSON parsing, state management, and bounded error recovery.

**Compute**: Designed for Google Colab T4 (free tier) or Kaggle GPU.

---
## Section 0 — Environment Setup

In [ ]:
# ── Step 1: Fix triton (must happen before bitsandbytes import) ───
!pip install -q --upgrade triton

# ── Step 2: Install bitsandbytes built for CUDA 12 / Python 3.12 ─
!pip install -q bitsandbytes --upgrade

# ── Step 3: Pin compatible versions of the rest of the stack ──────
!pip install -q \
    "transformers>=4.43.0" \
    "peft>=0.12.0" \
    "trl>=0.9.6" \
    "accelerate>=0.31.0" \
    "datasets>=2.20.0" \
    sentencepiece \
    protobuf \
    scipy

# ── Step 4: Verify bitsandbytes CUDA backend loads cleanly ────────
import subprocess, sys
result = subprocess.run(
    [sys.executable, '-c', 'import bitsandbytes as bnb; print("bnb CUDA:", bnb.COMPILED_WITH_CUDA)'],
    capture_output=True, text=True
)
print(result.stdout.strip() or result.stderr.strip())
print('✅ Installation complete')


Traceback (most recent call last):
  File "<string>", line 1, in <module>
AttributeError: module 'bitsandbytes' has no attribute 'COMPILED_WITH_CUDA'
✅ Installation complete


In [ ]:
# ── IMPORTANT: Restart the runtime after installation ─────────────
# bitsandbytes loads a native CUDA extension at import time.
# If you installed it in this session, you MUST restart before importing.
import os
print("Restarting Colab runtime to apply new packages...")
os.kill(os.getpid(), 9)  # Force restart


In [ ]:
import os, sys, re, json, textwrap, warnings, copy
import pandas as pd
import torch
from pathlib import Path
from typing import Optional

warnings.filterwarnings('ignore')

# ── GPU sanity check ───────────────────────────────────────────
print(f"CUDA available : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU            : {torch.cuda.get_device_name(0)}")
    print(f"VRAM           : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

CUDA available : True
GPU            : Tesla T4
VRAM           : 15.6 GB


In [ ]:
# ── Mount Google Drive (to save / load adapter weights) ─────────
from google.colab import drive
drive.mount('/content/drive')

DRIVE_DIR = Path('/content/drive/MyDrive/phase2_agent')
DRIVE_DIR.mkdir(parents=True, exist_ok=True)
print(f"Adapter save path: {DRIVE_DIR}")

Mounted at /content/drive
Adapter save path: /content/drive/MyDrive/phase2_agent


---
## Section 1 — Upload Project Files

Upload the following files from your assignment:
- `tool_executor.py`
- `sales_data.csv`
- `agent_trajectories_2k.json`

In [ ]:
from google.colab import files

print("Upload: tool_executor.py, sales_data.csv, agent_trajectories_2k.json")
uploaded = files.upload()
print(f"\nUploaded: {list(uploaded.keys())}")

Upload: tool_executor.py, sales_data.csv, agent_trajectories_2k.json


Saving agent_trajectories_2k.json to agent_trajectories_2k.json
Saving sales_data.csv to sales_data.csv
Saving tool_executor.py to tool_executor.py

Uploaded: ['agent_trajectories_2k.json', 'sales_data.csv', 'tool_executor.py']


In [ ]:
# ── Verify tool_executor loads correctly ─────────────────────────
sys.path.insert(0, '/content')
from tool_executor import ToolExecutor

df_sales = pd.read_csv('sales_data.csv')
print(f"Sales data shape : {df_sales.shape}")
print(f"Columns          : {list(df_sales.columns)}")
df_sales.head(3)

Sales data shape : (10000, 11)
Columns          : ['date', 'year', 'month', 'city', 'region', 'product', 'category', 'revenue', 'units_sold', 'cost', 'profit']


,date,year,month,city,region,product,category,revenue,units_sold,cost,profit
0,2023-12-06,2023,12,Kolkata,East,D,Electronics,15302.75,60,8177.77,7124.98
1,2022-04-12,2022,4,Delhi,North,D,Electronics,17626.77,32,11693.71,5933.06
2,2022-12-01,2022,12,Chennai,South,B,Clothing,10111.14,53,7306.30,2804.85


In [ ]:
# ── Load trajectory dataset ───────────────────────────────────────
with open('agent_trajectories_2k.json') as f:
    trajectories = json.load(f)

print(f"Trajectories loaded: {len(trajectories)}")
print("\nSample entry:")
print(json.dumps(trajectories[0], indent=2))

Trajectories loaded: 2000

Sample entry:
{
  "query": "What is total revenue for 2021?",
  "actions": [
    "filter_data(column='year', value=2021)",
    "aggregate_sum(column='revenue')"
  ]
}


---
## Section 2 — Tool Definitions & Execution Utilities

We define:
1. A string → dict action parser (re-used from `run_pipeline.py`)
2. A `run_actions()` helper that executes a list of string actions against the live DataFrame

In [ ]:
# ── Action string parser ──────────────────────────────────────────
def parse_agent_action(action_str: str) -> dict:
    """Convert a string action (e.g., filter_data(column='year', value=2022))
    into a ToolExecutor-compatible dict."""
    action_str = action_str.strip()

    if action_str.startswith("filter_data"):
        # Numeric value
        m = re.search(r"column='([^']+)',\s*value=(\d+)", action_str)
        if m:
            return {"tool": "filter", "args": {"column": m.group(1), "op": "==", "value": int(m.group(2))}}
        # String value
        m = re.search(r"column='([^']+)',\s*value='([^']+)'", action_str)
        if m:
            return {"tool": "filter", "args": {"column": m.group(1), "op": "==", "value": m.group(2)}}

    elif action_str.startswith("group_by"):
        m = re.search(r"column='([^']+)'", action_str)
        if m:
            return {"tool": "groupby", "args": {"column": m.group(1)}}

    elif action_str.startswith("aggregate_sum"):
        m = re.search(r"column='([^']+)'", action_str)
        if m:
            return {"tool": "aggregate", "args": {"column": m.group(1), "agg": "sum"}}

    elif action_str.startswith("aggregate_mean"):
        m = re.search(r"column='([^']+)'", action_str)
        if m:
            return {"tool": "aggregate", "args": {"column": m.group(1), "agg": "mean"}}

    elif action_str.startswith("aggregate_count"):
        m = re.search(r"column='([^']+)'", action_str)
        if m:
            return {"tool": "aggregate", "args": {"column": m.group(1), "agg": "count"}}

    elif action_str.startswith("sort_by"):
        m = re.search(r"column='([^']+)',\s*order='([^']+)'", action_str)
        if m:
            return {"tool": "sort", "args": {"column": m.group(1), "ascending": m.group(2) != 'desc'}}

    elif action_str.startswith("top_k"):
        m = re.search(r"k=(\d+)", action_str)
        if m:
            return {"tool": "topk", "args": {"k": int(m.group(1))}}

    raise ValueError(f"Cannot parse action: {action_str!r}")


def run_actions(action_strings: list, df: pd.DataFrame) -> str:
    """Execute a list of string actions on df, return the result as a string."""
    parsed = [parse_agent_action(a) for a in action_strings]
    executor = ToolExecutor(df)
    result_df = executor.execute(parsed)
    return result_df.to_string(index=False)


# ── Quick sanity test ─────────────────────────────────────────────
test_actions = ["filter_data(column='year', value=2022)", "aggregate_sum(column='revenue')"]
print(run_actions(test_actions, df_sales))

    revenue
44926739.48


---
## Section 3 — Dataset Construction for SFT

We build training samples in the **Thought → Action → Action Input** (ReAct) format that the model must learn to generate.

### Tool schema (system prompt context)
The system prompt describes the available tools to the model so it understands what actions it can take.

In [ ]:
# ── System prompt: tool descriptions ────────────────────────────────
SYSTEM_PROMPT = """\
You are a data analysis agent. You have access to the following tools that operate on a sales DataFrame with columns: date, year, month, city, region, product, category, revenue, units_sold, cost, profit.

Available tools:
1. filter_data(column='<col>', value=<val>)  — Filter rows where column equals value.
2. group_by(column='<col>')                  — Group data by the specified column.
3. aggregate_sum(column='<col>')             — Sum the specified column (use after group_by for per-group sums).
4. aggregate_mean(column='<col>')            — Mean of the specified column.
5. aggregate_count(column='<col>')           — Count rows / group sizes.
6. sort_by(column='<col>', order='desc'|'asc') — Sort results by column.
7. top_k(k=<n>)                              — Return the top-k rows.

IMPORTANT RULES:
- Always reason step-by-step using Thought → Action → Action Input format.
- After each Action Input, wait for the Observation before proceeding.
- When you have enough information to answer the question, output: Final Answer: <answer>
- Do NOT output anything outside this format.
"""

print(SYSTEM_PROMPT)

You are a data analysis agent. You have access to the following tools that operate on a sales DataFrame with columns: date, year, month, city, region, product, category, revenue, units_sold, cost, profit.

Available tools:
1. filter_data(column='<col>', value=<val>)  — Filter rows where column equals value.
2. group_by(column='<col>')                  — Group data by the specified column.
3. aggregate_sum(column='<col>')             — Sum the specified column (use after group_by for per-group sums).
4. aggregate_mean(column='<col>')            — Mean of the specified column.
5. aggregate_count(column='<col>')           — Count rows / group sizes.
6. sort_by(column='<col>', order='desc'|'asc') — Sort results by column.
7. top_k(k=<n>)                              — Return the top-k rows.

IMPORTANT RULES:
- Always reason step-by-step using Thought → Action → Action Input format.
- After each Action Input, wait for the Observation before proceeding.
- When you have enough information to 

In [ ]:
# ── Map action strings to ReAct format ──────────────────────────────
ACTION_THOUGHT_TEMPLATES = {
    'filter_data':      lambda a: f"I need to filter the data based on the condition specified.",
    'group_by':         lambda a: f"I need to group the data by the relevant column.",
    'aggregate_sum':    lambda a: f"I need to compute the sum of the target column.",
    'aggregate_mean':   lambda a: f"I need to compute the mean of the target column.",
    'aggregate_count':  lambda a: f"I need to count the entries.",
    'sort_by':          lambda a: f"I need to sort the results to rank them correctly.",
    'top_k':            lambda a: f"I need to select the top entries from the sorted results.",
}

def action_to_react_step(action_str: str) -> tuple[str, str, str]:
    """Returns (thought, action_name, action_input_json) for a single action string."""
    action_str = action_str.strip()
    tool_name = action_str.split('(')[0]
    thought = ACTION_THOUGHT_TEMPLATES.get(tool_name, lambda a: "I need to perform the next step.")(action_str)

    # Build action_input dict
    args_str = re.search(r'\((.*)\)', action_str)
    args_dict = {}
    if args_str:
        for part in re.findall(r"(\w+)=(['\"]?[^,)]+['\"]?)", args_str.group(1)):
            key = part[0]
            val = part[1].strip("'\"")
            # Try numeric conversion
            try:
                val = int(val)
            except ValueError:
                try:
                    val = float(val)
                except ValueError:
                    pass
            args_dict[key] = val

    return thought, tool_name, json.dumps(args_dict)


def trajectory_to_react_text(entry: dict, df: pd.DataFrame) -> str:
    """Convert one trajectory entry into a full ReAct conversation string."""
    query = entry['query']
    actions = entry['actions']

    lines = []
    lines.append(f"Human: {query}")
    lines.append("")
    lines.append("Assistant:")

    # Execute actions one-by-one to get real observations
    parsed_so_far = []
    prev_result = df

    for i, action_str in enumerate(actions):
        thought, action_name, action_input = action_to_react_step(action_str)
        lines.append(f"Thought: {thought}")
        lines.append(f"Action: {action_name}")
        lines.append(f"Action Input: {action_input}")

        # Execute up to this step to get the observation
        try:
            parsed_so_far.append(parse_agent_action(action_str))
            executor = ToolExecutor(df)
            result = executor.execute(parsed_so_far)
            if isinstance(result, pd.DataFrame):
                obs_text = result.to_string(index=False)[:500]  # Truncate large observations
            else:
                obs_text = str(result)[:500]
        except Exception as e:
            obs_text = f"Error: {e}"

        lines.append(f"Observation: {obs_text}")

    # Final answer — re-execute all actions and get the final result
    try:
        all_parsed = [parse_agent_action(a) for a in actions]
        executor = ToolExecutor(df)
        final_result = executor.execute(all_parsed)
        if isinstance(final_result, pd.DataFrame):
            answer = final_result.to_string(index=False)
        else:
            answer = str(final_result)
    except Exception as e:
        answer = f"Could not compute answer: {e}"

    lines.append(f"Thought: I now have all the information needed to answer the question.")
    lines.append(f"Final Answer: {answer}")

    return "\n".join(lines)


# ── Preview one sample ─────────────────────────────────────────────
sample_text = trajectory_to_react_text(trajectories[2], df_sales)
print(sample_text)

Human: What is total revenue for 2023?

Assistant:
Thought: I need to filter the data based on the condition specified.
Action: filter_data
Action Input: {"column": "year", "value": 2023}
Observation:       date  year  month      city region product    category  revenue  units_sold     cost   profit
2023-12-06  2023     12   Kolkata   East       D Electronics 15302.75          60  8177.77  7124.98
2023-03-21  2023      3    Mumbai   West       D Electronics 17335.93          26 11414.70  5921.22
2023-03-03  2023      3   Kolkata   East       D Electronics 13826.80          42  9325.09  4501.71
2023-12-24  2023     12 Bangalore  South       B    Clothing 13995.21          45  8983.46  5011.75

Thought: I need to compute the sum of the target column.
Action: aggregate_sum
Action Input: {"column": "revenue"}
Observation:     revenue
44410261.59
Thought: I now have all the information needed to answer the question.
Final Answer:     revenue
44410261.59


In [ ]:
# ── Build full training dataset ───────────────────────────────────
from datasets import Dataset
from tqdm import tqdm

print("Building training samples (this may take ~2 mins)...")

train_samples = []
errors = 0
print(len(trajectories))
for i, entry in tqdm(enumerate(trajectories)):
    try:
        text = trajectory_to_react_text(entry, df_sales)
        # Format as a complete chat: system + user/assistant turn
        full_text = f"{SYSTEM_PROMPT}\n\n{text}"
        train_samples.append({"text": full_text, "query": entry['query']})
    except Exception as e:
        errors += 1

print(f"✅ Built {len(train_samples)} samples ({errors} errors skipped)")

# 90/10 split
split_idx = int(0.9 * len(train_samples))
train_data = Dataset.from_list(train_samples[:split_idx])
eval_data  = Dataset.from_list(train_samples[split_idx:])

print(f"Train size : {len(train_data)}")
print(f"Eval size  : {len(eval_data)}")

Building training samples (this may take ~2 mins)...
2000


2000it [06:51,  4.87it/s]

✅ Built 2000 samples (0 errors skipped)
Train size : 1800
Eval size  : 200


---
## Part A — Supervised Fine-Tuning with QLoRA

We fine-tune **Mistral-7B-Instruct-v0.2** (or any compatible 7B model) using **QLoRA** (4-bit quantization + LoRA adapters) to fit within the T4's 16 GB VRAM.

In [ ]:
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    TrainingArguments,
    StoppingCriteria,
    StoppingCriteriaList,
)
from peft import LoraConfig, get_peft_model, TaskType, PeftModel
from trl import SFTTrainer

# ── Model choice ──────────────────────────────────────────────────
# Options (all fit on T4 with 4-bit):
#   "mistralai/Mistral-7B-Instruct-v0.2"   ← recommended
#   "meta-llama/Meta-Llama-3-8B-Instruct"  ← requires HF token
#   "Qwen/Qwen2-7B-Instruct"
MODEL_ID = "mistralai/Mistral-7B-Instruct-v0.2"

# If using LLaMA 3, set your token:
# HF_TOKEN = "hf_..."

print(f"Base model : {MODEL_ID}")

Base model : mistralai/Mistral-7B-Instruct-v0.2


In [ ]:
# ── 4-bit quantization config (QLoRA) ────────────────────────────
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
)

# ── Load tokenizer ────────────────────────────────────────────────
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'right'

print(f"Vocab size     : {tokenizer.vocab_size}")
print(f"EOS token      : {tokenizer.eos_token!r}")
print(f"Max model len  : {tokenizer.model_max_length}")

config.json:   0%|          | 0.00/596 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

Vocab size     : 32000
EOS token      : '</s>'
Max model len  : 1000000000000000019884624838656


In [ ]:
# ── Load base model (4-bit) ───────────────────────────────────────
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)
base_model.config.use_cache = False
base_model.config.pretraining_tp = 1

print("✅ Base model loaded")
print(f"Parameters: {sum(p.numel() for p in base_model.parameters()) / 1e9:.2f}B")

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

✅ Base model loaded
Parameters: 3.75B


In [ ]:
# ── LoRA adapter configuration ────────────────────────────────────
# Targeting the attention + FFN projection layers (standard for Mistral/LLaMA)
lora_config = LoraConfig(
    r=16,                          # LoRA rank
    lora_alpha=32,                 # Scaling factor
    target_modules=[               # Modules to adapt
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
)

model = get_peft_model(base_model, lora_config)
model.print_trainable_parameters()

trainable params: 41,943,040 || all params: 7,283,675,136 || trainable%: 0.5758


In [ ]:
# ── Training hyperparameters ──────────────────────────────────────
# Tuned for T4 16GB; reduce batch_size or max_seq_length if OOM occurs.
training_args = TrainingArguments(
    output_dir="./sft_checkpoints",
    num_train_epochs=1,
    per_device_train_batch_size=8,
    gradient_accumulation_steps=8,

    fp16=False,
    bf16=False,

    optim="adamw_torch",   # ✅ stable

    gradient_checkpointing=True,
    logging_steps=10,

    # evaluation_strategy="steps",
    # save_strategy="steps",
    eval_steps=50,
    save_steps=50,

    learning_rate=2e-4,
    max_grad_norm=0.3,
    warmup_steps=100,

    lr_scheduler_type="cosine",
    report_to="none",
)

# ── SFT Trainer ───────────────────────────────────────────────────
trainer = SFTTrainer(
    model=model,
    train_dataset=train_data,
    eval_dataset=eval_data,
    # peft_config=lora_config,
    # dataset_text_field="text",
    args=training_args,
)

print("✅ Trainer configured")

Adding EOS to train dataset:   0%|          | 0/1800 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/1800 [00:00<?, ? examples/s]

Adding EOS to eval dataset:   0%|          | 0/200 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/200 [00:00<?, ? examples/s]

✅ Trainer configured


In [ ]:
# ╔══════════════════════════════════════════════════════════╗
# ║  TRAINING  (skip this cell if loading saved weights)    ║
# ╚══════════════════════════════════════════════════════════╝
print("Starting SFT training...")
trainer.train()
print("\n✅ Training complete!")

Starting SFT training...


Step,Training Loss
10,1.479405


In [ ]:
# ── Save LoRA adapter weights to Google Drive ─────────────────────
ADAPTER_SAVE_PATH = str(DRIVE_DIR / 'lora_adapter')

model.save_pretrained(ADAPTER_SAVE_PATH)
tokenizer.save_pretrained(ADAPTER_SAVE_PATH)

print(f"✅ Adapter saved to: {ADAPTER_SAVE_PATH}")

### Loading Saved Weights (for Evaluation Sessions)

Run the cell below instead of training to load pre-saved adapter weights:

In [ ]:
# ── Load saved adapter (EVALUATION ONLY — skip if freshly trained) ─
ADAPTER_LOAD_PATH = str(DRIVE_DIR / 'lora_adapter')

base_model_eval = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)
model = PeftModel.from_pretrained(base_model_eval, ADAPTER_LOAD_PATH)
tokenizer = AutoTokenizer.from_pretrained(ADAPTER_LOAD_PATH)
tokenizer.pad_token = tokenizer.eos_token

model.eval()
print("✅ Adapter loaded and model set to eval mode")

---
## Part B — The ReAct Execution Engine

A pure-Python `while` loop that:
1. **Custom stopping criteria** — halts generation at `Observation:`
2. **Robust JSON parsing** — sanitises malformed model output
3. **State management & truncation** — prevents context window OOM
4. **Bounded error recovery** — self-correction up to `MAX_TURNS`
5. **Task completion** — detects `Final Answer:` and returns

In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  1. CUSTOM STOPPING CRITERIA                                    ║
# ║     Halt generation the moment the model predicts 'Observation:'║
# ╚══════════════════════════════════════════════════════════════════╝

class ObservationStoppingCriteria(StoppingCriteria):
    """
    Stops generation as soon as the model outputs the token sequence
    corresponding to 'Observation:'. This prevents the model from
    hallucinating tool outputs.
    """
    def __init__(self, stop_token_ids: list[list[int]]):
        self.stop_token_ids = stop_token_ids  # List of token-id sequences to stop on
        self._stop_flag = False

    def __call__(self, input_ids: torch.LongTensor, scores: torch.FloatTensor, **kwargs) -> bool:
        for stop_ids in self.stop_token_ids:
            stop_len = len(stop_ids)
            if input_ids.shape[1] >= stop_len:
                tail = input_ids[0, -stop_len:].tolist()
                if tail == stop_ids:
                    return True
        return False


def build_stopping_criteria(tokenizer) -> StoppingCriteriaList:
    """
    Encodes 'Observation:' in multiple possible ways (with/without newline,
    with/without leading space) to handle tokeniser differences.
    """
    stop_strings = [
        "\nObservation:",
        "Observation:",
        "\nObservation: ",
    ]
    all_stop_ids = []
    for s in stop_strings:
        ids = tokenizer.encode(s, add_special_tokens=False)
        if ids:
            all_stop_ids.append(ids)
    criteria = ObservationStoppingCriteria(all_stop_ids)
    return StoppingCriteriaList([criteria])


print("✅ Custom stopping criteria defined")

In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  2. ROBUST JSON / ACTION PARSER                                 ║
# ║     Handles trailing commas, markdown fences, missing brackets  ║
# ╚══════════════════════════════════════════════════════════════════╝

def sanitise_json(raw: str) -> str:
    """Strip markdown code fences and common JSON errors."""
    # Remove ```json ... ``` fences
    raw = re.sub(r'```(?:json)?\s*', '', raw)
    raw = raw.replace('```', '').strip()
    # Remove trailing commas before } or ]
    raw = re.sub(r',\s*([}\]])', r'\1', raw)
    # Attempt to close unclosed braces
    opens = raw.count('{') - raw.count('}')
    raw += '}' * max(0, opens)
    return raw


def parse_action_input(action_input_str: str) -> dict:
    """Robustly parse the Action Input line into a Python dict."""
    cleaned = sanitise_json(action_input_str.strip())
    try:
        return json.loads(cleaned)
    except json.JSONDecodeError:
        # Fallback: extract key=value pairs manually
        pairs = re.findall(r'"(\w+)"\s*:\s*("[^"]*"|\d+\.?\d*|true|false|null)', cleaned)
        result = {}
        for k, v in pairs:
            try:
                result[k] = json.loads(v)
            except Exception:
                result[k] = v.strip('"')
        if result:
            return result
        raise ValueError(f"Cannot parse Action Input: {action_input_str!r}")


def extract_react_fields(generated_text: str) -> dict:
    """
    Parse the model's generated text to extract:
      - thought       : str
      - action        : str (tool name)
      - action_input  : dict
      - final_answer  : str (if present)
    """
    result = {"thought": "", "action": None, "action_input": None, "final_answer": None}

    # Final Answer
    fa_match = re.search(r'Final Answer:\s*(.*)', generated_text, re.DOTALL | re.IGNORECASE)
    if fa_match:
        result["final_answer"] = fa_match.group(1).strip()
        return result

    # Thought
    t_match = re.search(r'Thought:\s*(.*?)(?=Action:|$)', generated_text, re.DOTALL | re.IGNORECASE)
    if t_match:
        result["thought"] = t_match.group(1).strip()

    # Action
    a_match = re.search(r'Action:\s*(\S+)', generated_text, re.IGNORECASE)
    if a_match:
        result["action"] = a_match.group(1).strip()

    # Action Input
    ai_match = re.search(r'Action Input:\s*(.*?)(?=Observation:|$)', generated_text, re.DOTALL | re.IGNORECASE)
    if ai_match:
        result["action_input"] = parse_action_input(ai_match.group(1).strip())

    return result


print("✅ Robust parser defined")

In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  3. TOOL DISPATCHER                                             ║
# ║     Convert parsed action_input dict → ToolExecutor action dict ║
# ╚══════════════════════════════════════════════════════════════════╝

# Maps model-generated action names to ToolExecutor tool names
TOOL_MAP = {
    "filter_data":      "filter",
    "group_by":         "groupby",
    "aggregate_sum":    "aggregate",
    "aggregate_mean":   "aggregate",
    "aggregate_count":  "aggregate",
    "sort_by":          "sort",
    "top_k":            "topk",
}

def dispatch_tool(action_name: str, action_input: dict, df: pd.DataFrame) -> tuple[str, pd.DataFrame]:
    """
    Execute one ReAct action step.
    Returns (observation_text, updated_df).
    """
    tool = TOOL_MAP.get(action_name)
    if tool is None:
        raise ValueError(f"Unknown action: {action_name!r}")

    # Build ToolExecutor args
    args = dict(action_input)
    if action_name == "filter_data":
        args.setdefault("op", "==")
    elif action_name == "aggregate_sum":
        args["agg"] = "sum"
    elif action_name == "aggregate_mean":
        args["agg"] = "mean"
    elif action_name == "aggregate_count":
        args["agg"] = "count"
    elif action_name == "sort_by":
        order = args.pop("order", "desc")
        args["ascending"] = (order != "desc")

    executor = ToolExecutor(df)
    result = executor.execute([{"tool": tool, "args": args}])

    if isinstance(result, pd.DataFrame):
        obs = result.to_string(index=False)
    else:
        obs = str(result)

    return obs, result if isinstance(result, pd.DataFrame) else df


print("✅ Tool dispatcher defined")

In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  4. STATE MANAGEMENT & TRUNCATION                               ║
# ║     Prevents context window OOM on large API observations       ║
# ╚══════════════════════════════════════════════════════════════════╝

MAX_OBSERVATION_CHARS = 600   # Truncate long tool outputs
MAX_CONTEXT_TOKENS    = 2048  # Truncate full context if too long

def truncate_observation(obs: str, max_chars: int = MAX_OBSERVATION_CHARS) -> str:
    """Truncate long observations and append a note."""
    if len(obs) <= max_chars:
        return obs
    return obs[:max_chars] + f"\n... [truncated to {max_chars} chars]"


def truncate_context(context: str, tokenizer, max_tokens: int = MAX_CONTEXT_TOKENS) -> str:
    """If the context exceeds max_tokens, trim from the middle (preserve start/end)."""
    tokens = tokenizer.encode(context, add_special_tokens=False)
    if len(tokens) <= max_tokens:
        return context
    # Keep first 30% and last 70%
    keep_start = int(max_tokens * 0.3)
    keep_end   = max_tokens - keep_start
    trimmed_ids = tokens[:keep_start] + tokens[-keep_end:]
    return tokenizer.decode(trimmed_ids, skip_special_tokens=True)


print("✅ Context management utilities defined")

In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  5. THE REACT EXECUTION LOOP                                    ║
# ║     Pure Python — no LangChain, LlamaIndex, AutoGen            ║
# ╚══════════════════════════════════════════════════════════════════╝

MAX_TURNS           = 5    # Maximum ReAct loop iterations
MAX_CONSECUTIVE_ERR = 2    # Consecutive failures before giving up

@torch.inference_mode()
def react_agent(
    query: str,
    df: pd.DataFrame,
    model,
    tokenizer,
    verbose: bool = True,
) -> str:
    """
    Run the multi-turn ReAct loop for a single user query.

    Returns the final answer string, or an error message if the loop
    exhausted its budget without reaching Final Answer.
    """
    model.eval()
    stopping_criteria = build_stopping_criteria(tokenizer)

    # ── Initialise context ─────────────────────────────────────────
    context = (
        f"{SYSTEM_PROMPT}\n\n"
        f"Human: {query}\n\n"
        f"Assistant:\n"
    )

    # Stateful: track the rolling DataFrame (starts as the full dataset)
    current_df = df.copy()
    turn        = 0
    consec_err  = 0

    if verbose:
        print(f"{'='*60}")
        print(f"Query: {query}")
        print(f"{'='*60}")

    # ── Main ReAct loop ────────────────────────────────────────────
    while turn < MAX_TURNS:
        turn += 1
        if verbose:
            print(f"\n--- Turn {turn} ---")

        # ① Truncate context if necessary
        safe_context = truncate_context(context, tokenizer)

        # ② Tokenise & generate (halt on 'Observation:')
        inputs = tokenizer(
            safe_context,
            return_tensors="pt",
            truncation=True,
            max_length=MAX_CONTEXT_TOKENS,
        ).to(model.device)

        with torch.no_grad():
            output_ids = model.generate(
                **inputs,
                max_new_tokens=256,
                do_sample=False,           # Greedy for determinism
                temperature=1.0,
                pad_token_id=tokenizer.eos_token_id,
                stopping_criteria=stopping_criteria,
            )

        # ③ Decode only newly generated tokens
        prompt_len    = inputs['input_ids'].shape[1]
        new_token_ids = output_ids[0, prompt_len:]
        generated     = tokenizer.decode(new_token_ids, skip_special_tokens=True)

        if verbose:
            print(f"[Model output]\n{generated.strip()}")

        # ④ Parse the generated text
        parsed = extract_react_fields(generated)

        # ── CASE A: Final Answer detected ──────────────────────────
        if parsed["final_answer"] is not None:
            final = parsed["final_answer"]
            if verbose:
                print(f"\n✅ Final Answer: {final}")
            return final

        # ── CASE B: Action + Action Input found ───────────────────
        action_name  = parsed.get("action")
        action_input = parsed.get("action_input")

        if action_name and action_input is not None:
            try:
                # ⑤ Execute the tool
                observation, current_df = dispatch_tool(action_name, action_input, current_df)
                observation = truncate_observation(observation)  # Prevent OOM
                consec_err  = 0  # Reset error counter on success

                if verbose:
                    print(f"[Observation]\n{observation}")

            except Exception as e:
                # ④ BOUNDED ERROR RECOVERY
                consec_err += 1
                observation = f"Error executing {action_name}: {str(e)}"
                current_df  = df.copy()  # Reset to original df on error

                if verbose:
                    print(f"[Tool Error] {observation} (consecutive errors: {consec_err})")

                if consec_err >= MAX_CONSECUTIVE_ERR:
                    if verbose:
                        print("[Loop] Max consecutive errors reached. Terminating.")
                    return f"Agent failed after {MAX_CONSECUTIVE_ERR} consecutive errors: {observation}"

            # ⑥ Append the observation to the context for the next turn
            context += generated
            if not generated.rstrip().endswith('Observation:'):
                context += "\nObservation:"
            context += f" {observation}\n"

        else:
            # Model produced neither a Final Answer nor a valid Action
            consec_err += 1
            observation = "Could not parse Action or Action Input. Please output Thought, Action, and Action Input correctly."
            context += generated + f"\nObservation: {observation}\n"

            if verbose:
                print(f"[Parse Error] {observation} (consecutive errors: {consec_err})")

            if consec_err >= MAX_CONSECUTIVE_ERR:
                return f"Agent failed: could not parse model output after {MAX_CONSECUTIVE_ERR} attempts."

    # ── Loop exhausted ─────────────────────────────────────────────
    if verbose:
        print(f"[Loop] Reached MAX_TURNS={MAX_TURNS} without Final Answer.")
    return f"Agent did not reach a final answer within {MAX_TURNS} turns."


print("✅ ReAct execution loop defined")

---
## Section 5 — Live Inference Demo

Run the agent on sample queries from the test set.

In [ ]:
# ── Run on a few sample queries ───────────────────────────────────
test_queries = [
    "What is total revenue for 2022?",
    "Top 3 cities by revenue in 2023",
    "What is the average profit by region?",
    "Total revenue for Electronics category",
]

for q in test_queries:
    answer = react_agent(q, df_sales, model, tokenizer, verbose=True)
    print(f"\n{'='*60}\n")

---
## Section 6 — Offline Tier-1 Accuracy Evaluation

Tier-1 evaluates whether the model generates the correct `Action` and `Action Input` sequence (without executing tools). We compare against the ground-truth action strings from the trajectory dataset.

In [ ]:
# ── Tier-1: Check action sequence accuracy ────────────────────────
def evaluate_tier1(entries: list, model, tokenizer, n_samples: int = 50) -> dict:
    """
    For each entry, generate the first Thought/Action/Action Input
    and check if the predicted action name matches ground truth.
    """
    correct = 0
    total   = 0
    results = []

    sample_entries = entries[-n_samples:]  # Use the held-out portion

    model.eval()
    stopping_criteria = build_stopping_criteria(tokenizer)

    for entry in sample_entries:
        query     = entry['query']
        gt_action = entry['actions'][0].split('(')[0]  # Ground truth: first action name

        prompt = (
            f"{SYSTEM_PROMPT}\n\n"
            f"Human: {query}\n\n"
            f"Assistant:\n"
        )

        inputs = tokenizer(
            prompt, return_tensors="pt", truncation=True, max_length=1024
        ).to(model.device)

        with torch.no_grad():
            output_ids = model.generate(
                **inputs,
                max_new_tokens=128,
                do_sample=False,
                pad_token_id=tokenizer.eos_token_id,
                stopping_criteria=stopping_criteria,
            )

        prompt_len = inputs['input_ids'].shape[1]
        generated  = tokenizer.decode(output_ids[0, prompt_len:], skip_special_tokens=True)
        parsed     = extract_react_fields(generated)
        pred_action = parsed.get("action", "") or ""

        is_correct = (pred_action.strip() == gt_action.strip())
        if is_correct:
            correct += 1
        total += 1

        results.append({
            "query": query,
            "ground_truth": gt_action,
            "predicted": pred_action,
            "correct": is_correct,
        })

    accuracy = correct / total * 100
    return {"accuracy": accuracy, "correct": correct, "total": total, "results": results}


# ── Run evaluation ─────────────────────────────────────────────────
eval_results = evaluate_tier1(trajectories, model, tokenizer, n_samples=50)

print(f"\nTier-1 Accuracy: {eval_results['accuracy']:.1f}%  ({eval_results['correct']}/{eval_results['total']})")
print("\nSample predictions:")
for r in eval_results['results'][:5]:
    status = "✅" if r['correct'] else "❌"
    print(f"  {status}  GT: {r['ground_truth']:20s}  | PRED: {r['predicted']:20s}  | Query: {r['query'][:60]}")

---
## Section 7 — Online Tier-2 Execution Evaluation

Tier-2 evaluates whether the full ReAct loop successfully reaches a `Final Answer` for each query.

In [ ]:
# ── Tier-2: Full end-to-end execution success rate ────────────────
def evaluate_tier2(entries: list, df: pd.DataFrame, model, tokenizer, n_samples: int = 20) -> dict:
    """
    Run the full ReAct loop on each query and check whether the agent
    successfully reaches a Final Answer.
    """
    success = 0
    total   = 0
    results = []

    sample_entries = entries[-n_samples:]

    for entry in sample_entries:
        query = entry['query']
        answer = react_agent(query, df, model, tokenizer, verbose=False)
        succeeded = not answer.startswith("Agent")
        if succeeded:
            success += 1
        total += 1
        results.append({"query": query, "answer": answer, "success": succeeded})
        print(f"{'✅' if succeeded else '❌'} {query[:70]}")

    return {"success_rate": success / total * 100, "success": success, "total": total, "results": results}


tier2_results = evaluate_tier2(trajectories, df_sales, model, tokenizer, n_samples=20)
print(f"\nTier-2 Success Rate: {tier2_results['success_rate']:.1f}%  ({tier2_results['success']}/{tier2_results['total']})")

---
## Section 8 — Bonus: Direct Preference Optimization (DPO)

DPO penalizes hallucinated/malformed trajectories and rewards clean ones. We construct preference pairs from the trajectory dataset:
- **Chosen**: correctly executed trajectory
- **Rejected**: the same query but with a hallucinated/wrong action

In [ ]:
from trl import DPOTrainer, DPOConfig

# ── Build preference pairs ─────────────────────────────────────────
def build_dpo_pairs(entries: list, df: pd.DataFrame) -> list:
    """
    For each trajectory:
      - chosen  : the ground-truth ReAct completion
      - rejected: a mutated version with an incorrect action
    """
    WRONG_ACTIONS = {
        "filter_data":    "group_by(column='city')",
        "group_by":       "filter_data(column='year', value=2020)",
        "aggregate_sum":  "aggregate_mean(column='revenue')",
        "aggregate_mean": "aggregate_sum(column='revenue')",
        "sort_by":        "aggregate_sum(column='revenue')",
        "top_k":          "sort_by(column='units_sold', order='asc')",
    }

    pairs = []
    for entry in entries:
        query   = entry['query']
        actions = entry['actions']
        prompt  = f"{SYSTEM_PROMPT}\n\nHuman: {query}\n\nAssistant:\n"

        # Chosen: correct trajectory
        chosen_steps = []
        for a in actions:
            t, an, ai = action_to_react_step(a)
            chosen_steps.append(f"Thought: {t}\nAction: {an}\nAction Input: {ai}")
        chosen_steps.append("Thought: I now have all information.\nFinal Answer: [computed]")
        chosen = "\n".join(chosen_steps)

        # Rejected: first action replaced with a wrong one
        first_tool = actions[0].split('(')[0]
        wrong_action = WRONG_ACTIONS.get(first_tool, "aggregate_sum(column='cost')")
        t_wrong, an_wrong, ai_wrong = action_to_react_step(wrong_action)
        rejected = (
            f"Thought: {t_wrong}\nAction: {an_wrong}\nAction Input: {ai_wrong}\n"
            "Thought: I have the answer.\nFinal Answer: [wrong]"
        )

        pairs.append({"prompt": prompt, "chosen": chosen, "rejected": rejected})

    return pairs


dpo_pairs = build_dpo_pairs(trajectories[:500], df_sales)  # Use subset for speed
dpo_dataset = Dataset.from_list(dpo_pairs)
print(f"DPO pairs: {len(dpo_dataset)}")
print("\nSample chosen:")
print(dpo_dataset[0]['chosen'][:300])

In [ ]:
# ── DPO Training ──────────────────────────────────────────────────
# Uses the SFT-finetuned model as the reference model

dpo_config = DPOConfig(
    output_dir="./dpo_checkpoints",
    num_train_epochs=1,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=5e-5,
    fp16=True,
    logging_steps=10,
    max_length=512,
    max_prompt_length=256,
    beta=0.1,              # KL penalty strength — higher = stay closer to reference
    report_to="none",
)

dpo_trainer = DPOTrainer(
    model=model,
    ref_model=None,          # None = use the PEFT base as reference
    args=dpo_config,
    train_dataset=dpo_dataset,
    tokenizer=tokenizer,
)

print("Starting DPO training...")
dpo_trainer.train()
print("✅ DPO training complete")

# Save DPO-refined adapter
DPO_SAVE_PATH = str(DRIVE_DIR / 'dpo_adapter')
model.save_pretrained(DPO_SAVE_PATH)
tokenizer.save_pretrained(DPO_SAVE_PATH)
print(f"DPO adapter saved to: {DPO_SAVE_PATH}")

---
## Section 9 — Held-Out Test Queries (Evaluation Slot)

During the live presentation, your evaluator will provide test queries. Run this cell with their queries.

In [ ]:
# ── Upload the held-out CSV if it differs from training data ──────
print("Upload the held-out sales CSV (or press Cancel to reuse training CSV)")
try:
    uploaded_test = files.upload()
    test_csv_name = list(uploaded_test.keys())[0]
    df_test = pd.read_csv(test_csv_name)
    print(f"Loaded test CSV: {test_csv_name} — {df_test.shape}")
except Exception:
    df_test = df_sales.copy()
    print("Using training CSV as test data.")

In [ ]:
# ── Live inference on evaluator queries ───────────────────────────
EVALUATOR_QUERIES = [
    # Replace these with the evaluator's actual test queries
    "What is the total revenue for the year 2023?",
    "Which region has the highest average profit?",
    "Top 5 products by units sold in 2022",
]

for query in EVALUATOR_QUERIES:
    answer = react_agent(query, df_test, model, tokenizer, verbose=True)
    print(f"\n>>> Final Answer: {answer}")
    print("\n" + "="*60 + "\n")

---
## Summary

| Component | Status |
|-----------|--------|
| QLoRA SFT on ToolAlpaca-format data | ✅ |
| Custom `Observation:` stopping criteria | ✅ |
| Robust JSON parser (trailing commas, fences) | ✅ |
| State management & context truncation | ✅ |
| Bounded error recovery (max 2 consecutive errors) | ✅ |
| `Final Answer:` detection & loop break | ✅ |
| Tier-1 offline accuracy evaluation | ✅ |
| Tier-2 online execution evaluation | ✅ |
| Bonus: DPO preference alignment | ✅ |
| Pure Python — no LangChain/LlamaIndex | ✅ |